In [1]:
import chromadb
import os
from dotenv import load_dotenv
from chromadb.utils import embedding_functions
import yfinance as yf
from bs4 import BeautifulSoup
import requests
import re
import time
from langchain_text_splitters import RecursiveCharacterTextSplitter
from nltk.tokenize import sent_tokenize
import pysbd

/home/dave/Development/UTS/nlp-at2/.venv/lib/python3.13/site-packages/pysbd/segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
/home/dave/Development/UTS/nlp-at2/.venv/lib/python3.13/site-packages/pysbd/lang/arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
/home/dave/Development/UTS/nlp-at2/.venv/lib/python3.13/site-packages/pysbd/lang/persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


Connect to the Chroma Vector DB

In [2]:
# Path to .env location containing the Chroma API keys
env_path = os.path.join("../data", ".env")

# Parse the .env file and retrieve the API keys
if load_dotenv(env_path):
    print("✅ Environment variables loaded from data/.env")
else:
    print("❌ Failed to load data/.env - Check if the file exists!")

CF_CLIENT_ID = os.getenv("CF-ACCESS-CLIENT-ID")
CF_CLIENT_SECRET = os.getenv("CF-ACCESS-CLIENT-SECRET")
CHROMA_URL = os.getenv("CHROMA_URL")

print(f"ID: {CF_CLIENT_ID}")
print(f"Secret: {CF_CLIENT_SECRET}")
print(f"Chroma URL: {CHROMA_URL}")


✅ Environment variables loaded from data/.env
ID: 15703ac0e5797dac885c851f20afe6cb.access
Secret: 9db6c201a502a20d0aa9eae4dac06d36b2f6a29b46201c20df2f5174263ddf57
Chroma URL: chroma.taskcomply.com


In [3]:
# 2. Setup the Client with Custom Headers
client = chromadb.HttpClient(
    host=CHROMA_URL,                              # Your Cloudflare URL
    port=443,                                     # Standard HTTPS port
    ssl=True,                                     # MUST be True for Cloudflare
    headers={
        "CF-Access-Client-Id": CF_CLIENT_ID,
        "CF-Access-Client-Secret": CF_CLIENT_SECRET
    },
)

# 3. Test the connection
print(f"Connection Heartbeat: {client.heartbeat()}")

Connection Heartbeat: 1775903796110476843


Determine embedding function and create 2 streams:
- Fundamentals
- Sentiment

In [4]:
# 4. Use a lightweight embedding function
# Default is 'all-MiniLM-L6-v2' which is great for financial headlines
emb_fn = embedding_functions.DefaultEmbeddingFunction()

# 3. Create the two streams
fundamentals_col = client.get_or_create_collection(
    name="stream1_fundamentals",
    embedding_function=emb_fn
)

sentiment_col = client.get_or_create_collection(
    name="stream2_sentiment",
    embedding_function=emb_fn
)

In [5]:
collections = client.list_collections()

print(collections)

[Collection(name=stream1_fundamentals), Collection(name=stream2_sentiment)]


Remove old experimental data

In [6]:
# 1. Wipe the old collections
for col_name in ["stream1_fundamentals", "stream2_sentiment"]:
    try:
        client.delete_collection(name=col_name)
        print(f"🔥 Deleted collection: {col_name}")
    except Exception as e:
        print(f"ℹ️ {col_name} didn't exist or already deleted.")

# 2. Re-initialize them fresh
emb_fn = chromadb.utils.embedding_functions.DefaultEmbeddingFunction()

fundamentals_col = client.get_or_create_collection(name="stream1_fundamentals", embedding_function=emb_fn)
sentiment_col = client.get_or_create_collection(name="stream2_sentiment", embedding_function=emb_fn)

print("✅ Server collections are reset and ready for the clean run.")


🔥 Deleted collection: stream1_fundamentals
🔥 Deleted collection: stream2_sentiment
✅ Server collections are reset and ready for the clean run.


In [7]:

col = client.get_collection(name="stream2_sentiment")
print(col.count())


0


2. Stream 2: Ingesting Sentiment (The "Market Pulse")
For the sentiment stream, we want the most recent headlines. Since Tesla (TSLA) just had a delivery miss on April 2nd, and Apple (AAPL) just celebrated its 50th anniversary on April 1st, there is plenty of fresh data.

In [8]:
# The "Discovery" List
stocks_to_track = [
    {"ticker": "NVDA", "name": "Nvidia"},
    {"ticker": "TSLA", "name": "Tesla"},
    {"ticker": "AAPL", "name": "Apple"}
]

In [9]:

def get_verified_article_data(url, target_ticker, company_name):
    """
    Final Master's Level Scraper:
    1. Full-page scan for ticker/name density.
    2. 'Anchor Zone' check (Nvidia must be in the first 20% or the Title).
    3. 'Stop Marker' extraction to prevent appended articles.
    """
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code != 200:
            return None

        soup = BeautifulSoup(response.text, 'html.parser')

        # 1. CLEAN THE ENTIRE SOUP (Remove non-article noise)
        for noise in soup.find_all(["script", "style", "nav", "footer", "aside", "header"]):
            noise.decompose()

        # 2. FULL BODY TEXT SCAN
        entire_page_text = soup.get_text(separator=" ").upper()

        # 3. DENSITY & ANCHOR CHECK
        # How many times is our stock mentioned?
        ticker_count = entire_page_text.count(target_ticker.upper())
        name_count = entire_page_text.count(company_name.upper())
        total_mentions = ticker_count + name_count

        # Where is it mentioned? (Kill the 'Bitcoin' comparison articles)
        anchor_zone = entire_page_text[:int(len(entire_page_text) * 0.20)]
        page_title = soup.find('title').get_text().upper() if soup.find('title') else ""

        # RELEVANCE LOGIC:
        # Pass if in Title OR (Mentioned early AND at least 3 times total)
        is_relevant = False
        if target_ticker.upper() in page_title or company_name.upper() in page_title:
            is_relevant = True
        elif (target_ticker.upper() in anchor_zone or company_name.upper() in anchor_zone) and total_mentions >= 3:
            is_relevant = True

        if not is_relevant:
            return None

        # 4. EXTRACTION (Finding the 'Meat' of the article)
        # We retain the original test and caps of the article for finBet as it is case sensitive.
        best_container = None
        max_p = 0
        for div in soup.find_all(['div', 'article', 'main', 'section']):
            p_count = len(div.find_all('p', recursive=False))
            if p_count > max_p:
                max_p = p_count
                best_container = div

        if best_container:
            # 5. STOP MARKER LOGIC (Prevent 'Read Next' bleeding)
            stop_markers = ["read-next", "related-stories", "recirc-feed", "caas-footer", "article-separator"]
            clean_elements = []

            for el in best_container.find_all(['p', 'h1', 'h2', 'h3']):
                # Check class of element and its immediate parent
                el_classes = " ".join(el.get('class', [])) + " " + " ".join(el.parent.get('class', []))

                if any(marker in el_classes.lower() for marker in stop_markers):
                    break # Stop here: we've hit the appended article feed

                txt = el.get_text().strip()
                if len(txt) > 30: # Filter out social buttons/byline scraps
                    clean_elements.append(txt)

            full_text = "\n\n".join(clean_elements)

            # Final sanity check: Is it an actual article (>400 chars)?
            return full_text if len(full_text) > 400 else None

        return None
    except Exception:
        return None

In [10]:
# 1. THE SHARED ENGINE
def get_unified_news_pool(ticker_sym, company_name):
    """Exact same discovery logic for both Inspection and Writing."""
    # Standardize the queries here!
    search_queries = [
        ticker_sym,
        company_name,
        f"{company_name} stock"
    ]

    pool = {}
    for q in search_queries:
        # Requesting 25 to ensure we get a deep overlap
        results = yf.Search(q, news_count=25).news
        for item in results:
            pool[item['uuid']] = item

    print(f"🔎 Total unique articles discovered: {len(pool)}")
    print("-" * 50)

    return pool

Initial check of scraped articles before writing to ChromaDB

In [11]:
def inspect_discovered_news(ticker_sym, company_name):

    # 1. Shared search
    discovered_pool = get_unified_news_pool(ticker_sym, company_name)

    # 2. Verification & Staging Phase
    verified_articles = []

    for uuid, item in discovered_pool.items():
        print(f"UUID: {uuid}")
        print(f"URL: {item['link']}")

        # Fetch and verify
        content = get_verified_article_data(item['link'], ticker_sym, company_name)

        if content:
            # Identify if it's a "Frankenstein" (likely appended articles)
            word_count = len(content.split())
            is_long = "⚠️ (Long/Possible Append)" if word_count > 1500 else ""

            verified_articles.append({
                "uuid": uuid,
                "headline": item['title'],
                "content": content,  # Keeping full content for your FinBERT later
                "body_snippet": content[:300].replace('\n', ' ') + "...",
                "word_count": word_count,
                "related": item.get('relatedTickers', []),
                "url": item['link']
            })
            print(f"✅ Verified: {item['title'][:50]}... [{word_count} words] {is_long}")
        else:
            print(f"❌ Rejected: {item['title'][:50]}...")

        time.sleep(0.4) # Respecting 2026 API thresholds

    # Return the full list of verified articles and the original pool
    return verified_articles, discovered_pool



In [12]:
# --- EXECUTION ---
# target = "NVDA"
# name = "Nvidia"
for stock in stocks_to_track:
    staged_data, original_pool = inspect_discovered_news(stock['ticker'], stock['name'])

    # --- THE "DEEP INSPECTION" VIEW ---
    print(f"\n--- INSPECTION REPORT: {len(staged_data)} VERIFIED ARTICLES ---")
    for i, art in enumerate(staged_data):
        print(f"\n[{i+1}] {art['headline']} ({art['word_count']} words)")
        print(f"    URL: {art['url']}")
        print(f"    Related: {art['related']}")
        print(f"    Preview: {art['body_snippet']}")
        print("-" * 30)

🔎 Total unique articles discovered: 21
--------------------------------------------------
UUID: bdbd1ae8-052b-31e7-aa04-bc015a6f942c
URL: https://finance.yahoo.com/markets/stocks/articles/nvidia-corporation-nvda-israel-englander-142908548.html
✅ Verified: NVIDIA Corporation (NVDA): Israel Englander Is Lon... [341 words] 
UUID: e075a871-e08a-3c89-924b-766c1a9ec5d3
URL: https://finance.yahoo.com/markets/stocks/articles/market-chatter-nvidia-chips-allegedly-191018064.html
❌ Rejected: Market Chatter: Nvidia Chips Allegedly Brought Int...
UUID: a5e41c3f-8f32-3fe0-8e17-df34c6198b1e
URL: https://finance.yahoo.com/markets/stocks/articles/wall-street-analysts-see-46-135503544.html
✅ Verified: Wall Street Analysts See a 46.16% Upside in Nvidia... [277 words] 
UUID: 4c982450-23bd-3885-a694-b3f1348497ab
URL: https://finance.yahoo.com/m/4c982450-23bd-3885-a694-b3f1348497ab/stocks-rallied-after.html
✅ Verified: Stocks Rallied After President Donald Trump's Ceas... [522 words] 
UUID: 1b4782f9-bf7d-36

Check all articles are unique before writing to ChromaDB

In [13]:
all_headlines = [art['headline'] for art in staged_data]
unique_headlines = set(all_headlines)

print(f"Total Articles: {len(all_headlines)}")
print(f"Unique Headlines: {len(unique_headlines)}")

if len(all_headlines) != len(unique_headlines):
    print("⚠️ Warning: You have duplicate stories in your staging area!")

Total Articles: 12
Unique Headlines: 12


Write to Chroma DB

## Chunk Articles in ChromaDB for finBERT

### Recursive Character Splitter

In [14]:
def save_to_chroma_recursive_chunked(collection, uuid, text, item, ticker, quality="high"):
    splitter = RecursiveCharacterTextSplitter(
        # 1. Define the Splitter
        # chunk_size: ~1024 chars is good for FinBERT (~250 words)
        # chunk_overlap: Keep 200 chars so context isn't lost between clips
        chunk_size=1024,
        # NVIDIA tests on FinanceBench with 10,24 token chunk found 15% to be optimal
        # https://developer.nvidia.com/blog/finding-the-best-chunking-strategy-for-accurate-ai-responses/
        chunk_overlap=154
    )

    chunks = splitter.split_text(text)

    if not chunks:
        print(f"⚠️ No chunks generated for {ticker}: {item.get('title', 'unknown title')}")
        return

    chunk_ids = []
    chunk_metadatas = []

    for idx, chunk in enumerate(chunks):
        chunk_id = f"{uuid}-{idx}"
        meta = {
            "ticker": ticker,
            "title": item.get("title", ""),
            "link": item.get("link", ""),
            "publisher": item.get("publisher", ""),
            "published": item.get("providerPublishTime", ""),
            "quality": quality,
            "chunk_index": idx,
            "chunk_count": len(chunks),
            "original_uuid": uuid,
        }

        chunk_ids.append(chunk_id)
        chunk_metadatas.append(meta)

    if not (len(chunks) == len(chunk_ids) == len(chunk_metadatas)):
        raise ValueError(
            f"Chunk insertion mismatch: documents={len(chunks)}, "
            f"ids={len(chunk_ids)}, metadatas={len(chunk_metadatas)}"
        )

    collection.add(
        documents=chunks,
        metadatas=chunk_metadatas,
        ids=chunk_ids
    )

### Sentence level chunking

We employ a sliding window sentence chunking strategy to preserve local semantic context while maintaining sentiment granularity, improving the effectiveness of both retrieval and sentiment classification.

*"Sentence-Level Sliding Window Vectorisation"* increases the training sample size by over 5x from recursive chunking without needing new source articles.

In [15]:
def sliding_window(sentences, window_size=2):
    chunks = []

    for i in range(len(sentences) - window_size + 1):
        chunk = " ".join(sentences[i:i + window_size])
        chunks.append(chunk)

    return chunks

In [16]:
def save_to_chroma_sentence_sliding(collection, uuid, text, item, ticker, quality="high", window_size=2):
    # 1. Initialise the Financial-Aware Segmenter
    segmenter = pysbd.Segmenter(language="en", clean=False)

    # 2. Segment using pySBD instead of sent_tokenize
    # This will correctly handle "Nvidia Corp. announced..." as one sentence.
    sentences = segmenter.segment(text)

    # 3. Clean and Filter
    sentences = [s.strip() for s in sentences if len(s.split()) >= 3]

    # --- FALLBACK FOR SHORT ARTICLES (Headlines/Barron's Paywalls) ---
    if len(sentences) < window_size:
        # If we have only 1 sentence, we still want it!
        chunks = [" ".join(sentences)] if sentences else [item.get("title", "")]
    else:
        # Proceed with your sliding window logic
        chunks = sliding_window(sentences, window_size)

    chunk_ids = []
    chunk_metadatas = []

    for idx, chunk in enumerate(chunks):
        chunk_id = f"{uuid}-{idx}"

        meta = {
            "ticker": ticker,
            "title": item.get("title", ""),
            "link": item.get("link", ""),
            "publisher": item.get("publisher", ""),
            "published": item.get("providerPublishTime", ""),
            "quality": quality,
            "chunk_index": idx,
            "chunk_count": len(chunks),
            "original_uuid": uuid,
            "window_size": window_size,
        }

        chunk_ids.append(chunk_id)
        chunk_metadatas.append(meta)

    if not chunk_ids:
        return

    collection.add(
        documents=chunks,
        metadatas=chunk_metadatas,
        ids=chunk_ids
    )

In [17]:
def get_bulk_verified_news_and_write_to_chromadb(ticker_sym, company_name, collection):
    discovered_news = get_unified_news_pool(ticker_sym, company_name)
    print(f"🔎 Pool Size for {ticker_sym}: {len(discovered_news)}")

    for uuid, item in discovered_news.items():
        # Look for the UUID inside the metadata 'original_uuid' field
        existing = collection.get(
            where={"original_uuid": uuid},
            limit=1
        )
        if existing['ids']:
            # print(f"⏩ Already in DB: {item['title'][:30]}")
            continue

        # --- STEP 1: Attempt the Surgical Scrape ---
        full_content = get_verified_article_data(item['link'], ticker_sym, company_name)

        if full_content:
            # TIER 1: SUCCESS
            save_to_chroma_sentence_sliding(collection, uuid, full_content, item, ticker_sym, quality="high", window_size=2)
            print(f"✅ Full Text: {item['title'][:50]}...")

        else:
            # --- STEP 2: Try for a Summary ---
            summary = item.get('summary') or item.get('description')

            if summary and len(summary) > 60:
                # TIER 2: SUCCESS
                save_to_chroma_sentence_sliding(collection, uuid, summary, item, ticker_sym, quality="medium", window_size=2)
                print(f"⚠️ Summary Only: {item['title'][:40]}...")

            else:
                # --- STEP 3: The "Headline Upcycle" (The Fix for Barron's and other pay-walled publications) ---
                headline = item.get('title', 'No Headline')
                pub = item.get('publisher', 'Unknown Source')
                related = ", ".join(item.get('relatedTickers', []))

                # We build a 'Synthetic' document so the embedding model has context
                synthetic_doc = f"Financial News Headline from {pub}: {headline}. Related Assets: {related}."

                save_to_chroma_sentence_sliding(collection, uuid, synthetic_doc, item, ticker_sym, quality="low", window_size=1)
                print(f"💡 Headline Only (Paywalled/Empty): {headline[:50]}...")

        time.sleep(0.5)



In [18]:
# Write to Chroma DB
for stock in stocks_to_track:
    print(f"\n--- Harvesting {stock['ticker']} ---")
    get_bulk_verified_news_and_write_to_chromadb(stock['ticker'], stock['name'], sentiment_col)



--- Harvesting NVDA ---
🔎 Total unique articles discovered: 21
--------------------------------------------------
🔎 Pool Size for NVDA: 21
✅ Full Text: NVIDIA Corporation (NVDA): Israel Englander Is Lon...
💡 Headline Only (Paywalled/Empty): Market Chatter: Nvidia Chips Allegedly Brought Int...
✅ Full Text: Wall Street Analysts See a 46.16% Upside in Nvidia...
✅ Full Text: Stocks Rallied After President Donald Trump's Ceas...
✅ Full Text: Why Broadcom Stock Is a Better Long-Term AI Stock ...
✅ Full Text: The AI Stock Wall Street Can't Stop Talking About ...
✅ Full Text: Want to Invest in OpenAI Before Its Blockbuster IP...
✅ Full Text: Here's My Top Artificial Intelligence (AI) Stock f...
✅ Full Text: Is Cryptocurrency a Legitimate Part of a Long-Term...
💡 Headline Only (Paywalled/Empty): Could We Be Looking at a Worst-Case Scenario for t...
✅ Full Text: Nvidia Is Nearly The Same Price as the S&P 500. It...
💡 Headline Only (Paywalled/Empty): Nvidia Stock Is on an 8-Day Winning Streak. 

In [19]:
col = client.get_collection(name="stream2_sentiment")
print(f"Total Articles successfully written to ChromaDB: {col.count()}")

Total Articles successfully written to ChromaDB: 957


In [20]:
# 1. Get the total chunk count
total_chunks = sentiment_col.count()

# 2. Get the unique article count (using metadata)
all_meta = sentiment_col.get(include=["metadatas"])
unique_uuids = set([m['original_uuid'] for m in all_meta['metadatas']])

print(f"📊 --- DATA DENSITY REPORT ---")
print(f"Total Chunks: {total_chunks}")
print(f"Unique Articles: {len(unique_uuids)}")
print(f"Average Chunks Per Article: {total_chunks / len(unique_uuids):.2f}")

# 3. Check for specific Ticker distribution
for t in ["NVDA", "AAPL", "TSLA"]:
    count = len([m for m in all_meta['metadatas'] if m['ticker'] == t])
    print(f"📍 {t} Chunks: {count}")

📊 --- DATA DENSITY REPORT ---
Total Chunks: 957
Unique Articles: 62
Average Chunks Per Article: 15.44
📍 NVDA Chunks: 321
📍 AAPL Chunks: 299
📍 TSLA Chunks: 337


## Sentiment Analysis - finBERT

We employ FinBERT, a domain-specific sentiment model trained on financial text, and do not perform additional fine-tuning due to limited labelled data and project scope constraints. Instead, we focus on improving sentiment quality through retrieval-based filtering and aggregation.